# AI Lab - Practical Question No. 6
## Multi-Agent Environment Simulation: Builder vs. Destroyer in a 4x4 Grid

- **Syllabus Area**: Intelligent Agents
- **Difficulty**: 🟠 Advanced
- **Practical Type**: Simulation
- **Core AI Concept**: Multi-Agent Systems, Competitive/Cooperative Environments, Dynamic & Sequential Worlds
- **Python Concepts Tested**: OOP, shared references, state modification, NumPy matrix manipulation
- **Complexity**: Time Complexity: $O(S)$ where $S$ is steps | Space Complexity: $O(\text{Grid Size}) = O(16) = O(1)$

---

### Problem Statement
Implement a multi-agent simulation in Python where two autonomous agents operate in the same 4x4 grid.
Agent A is a **'Builder'** that places blocks, and Agent B is a **'Destroyer'** that removes them.
Simulate their interaction over 10 steps and display the final grid state.

**Task Requirements**:
1. Create two separate Agent objects.
2. Define their specific percept-action rules.
3. Manage a shared environment object.
4. Execute their turns sequentially.

**Input Requirements**: Initial 4x4 empty grid.
**Expected Output**: Grid state after 10 steps of interaction.
**Constraints / Conditions**: Agents start at opposite corners: $(0, 0)$ and $(3, 3)$.
**Sample Input**: Steps: 10
**Sample Output**: Matrix displaying 1s (blocks) and 0s (empty space).

**Optional Challenge**: Make the agents aware of each other's location to actively avoid collisions.

### Agent PEAS Specification

| PEAS Component | Builder Agent (Agent A) | Destroyer Agent (Agent B) |
| :--- | :--- | :--- |
| **P**erformance Measure | Maximize blocks placed and retained ($U_A = +1 \times \text{blocks}$) | Maximize blocks removed ($U_B = -1 \times \text{blocks} = +1 \times \text{empty}$) |
| **E**nvironment | Shared 4x4 discrete grid with cell values $\{0, 1\}$ | Shared 4x4 discrete grid with cell values $\{0, 1\}$ |
| **A**ctuators | `PlaceBlock`, `MoveUp`, `MoveDown`, `MoveLeft`, `MoveRight`, `NoOp` | `RemoveBlock`, `MoveUp`, `MoveDown`, `MoveLeft`, `MoveRight`, `NoOp` |
| **S**ensors | Grid coordinate sensor, local cell inspection (0 vs 1), peer coordinate | Grid coordinate sensor, local cell inspection (0 vs 1), peer coordinate |

### Environmental Properties (Russell & Norvig Framework)
- **Multi-Agent**: Competitive (Zero-Sum) environment where agents have strictly opposing utility functions.
- **Sequential**: Decisions made at step $t$ directly impact subsequent states, percepts, and agent choices.
- **Dynamic**: From each agent's local viewpoint, the world changes between its turns due to the actions of the adversary.
- **Discrete**: Discrete coordinates $\{(r, c) \mid 0 \le r, c < 4\}$ and discrete turn-based time steps.

In [ ]:
# Setup environment and import core modules
import sys
from pathlib import Path

# Add repository root to Python path
REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from src.multi_agent_simulation import (
    BuilderAgent,
    DestroyerAgent,
    MultiAgentGridEnvironment,
    run_multi_agent_simulation,
    CELL_EMPTY,
    CELL_BLOCK,
    ROLE_BUILDER,
    ROLE_DESTROYER,
    TURN_MODE_COMPOUND,
    TURN_MODE_DISCRETE,
)

print("Packages and simulation modules successfully loaded!")

### Step 1: Initialize Separate Agent Objects and Shared Environment

We create two separate agent instances inheriting from `BaseAgent`:
- `Agent A` (Builder) at corner `(0, 0)`.
- `Agent B` (Destroyer) at opposite corner `(3, 3)`.

Both agents share a reference to a single `MultiAgentGridEnvironment` instance containing an initial 4x4 empty grid (all 0s).

In [ ]:
# Initialize agents at opposite corners
builder = BuilderAgent(name="Agent_A", initial_position=(0, 0), seed=42)
destroyer = DestroyerAgent(name="Agent_B", initial_position=(3, 3), seed=1042)

# Initialize shared 4x4 environment
env = MultiAgentGridEnvironment(rows=4, cols=4, agent_a=builder, agent_b=destroyer)

print("Initial 4x4 Grid Matrix:")
print(env.grid)
print(f"\nAgent A: {builder.name} ({builder.role}) at {builder.position}")
print(f"Agent B: {destroyer.name} ({destroyer.role}) at {destroyer.position}")

### Step 2: Execute 10-Step Sequential Multi-Agent Simulation

We run the simulation loop for 10 sequential rounds.
In each round, Agent A executes its percept-action rule, mutating the shared grid,
followed by Agent B perceiving the updated grid and executing its percept-action rule.

In [ ]:
# Execute 10 steps of interaction
result = run_multi_agent_simulation(
    steps=10,
    grid_size=(4, 4),
    seed=42,
    avoid_collisions=False,
    step_mode=TURN_MODE_COMPOUND,
)

print("Sample Output: Matrix displaying 1s (blocks) and 0s (empty space):")
print(result.display_matrix())
print()
print(result.summary())

### Step 3: Chronological Step Trace Inspection

Let us observe the turn-by-turn trace showing how blocks were placed and removed over the 10 steps.

In [ ]:
print("Detailed Turn-by-Turn Interaction Trace:")
print("-" * 75)
for record in result.history:
    print(record.format_trace())
print("-" * 75)

### Step 4: Visual Heatmap of the Final Grid State

We visualize the final 4x4 matrix using Matplotlib, showing block occupancy and the final coordinates of both agents.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
grid = result.final_grid

# Custom colormap: 0 = Light Gray (Empty), 1 = Slate Blue (Block)
cmap = mcolors.ListedColormap(["#f0f3f6", "#3b82f6"])
ax.imshow(grid, cmap=cmap, vmin=0, vmax=1)

# Draw gridlines
ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
ax.grid(which="minor", color="#64748b", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", size=0)

# Annotate cells
for r in range(4):
    for c in range(4):
        val = grid[r, c]
        label = f"{val}"
        if (r, c) == result.agent_a_final:
            label += "\n[Agent A]"
        if (r, c) == result.agent_b_final:
            label += "\n[Agent B]"
        text_color = "white" if val == 1 else "#1e293b"
        ax.text(c, r, label, ha="center", va="center", fontsize=11, fontweight="bold", color=text_color)

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels([f"Col {c}" for c in range(4)])
ax.set_yticklabels([f"Row {r}" for r in range(4)])
ax.set_title("Multi-Agent 4x4 Grid State after 10 Steps", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

### Step 5: Optional Challenge: Collision Awareness & Active Avoidance

We now enable collision avoidance (`avoid_collisions=True`).
When agents detect their peer occupying an adjacent cell, they prune that move from candidate transitions,
falling back to `NoOp` if cornered, guaranteeing zero physical collisions.

In [ ]:
# Run simulation with Collision Avoidance enabled
result_avoidance = run_multi_agent_simulation(
    steps=10,
    grid_size=(4, 4),
    seed=42,
    avoid_collisions=True,
    step_mode=TURN_MODE_COMPOUND,
)

print("Comparison of Collision Statistics:")
print(f"Without Avoidance -> Collisions Occurred: {result.collisions_occurred}, Avoided: {result.collisions_avoided_count}")
print(f"With Avoidance    -> Collisions Occurred: {result_avoidance.collisions_occurred}, Avoided: {result_avoidance.collisions_avoided_count}")
print("\nFinal Matrix with Collision Avoidance:")
print(result_avoidance.display_matrix())

### Step 6: Viva Voce Reference & Conceptual Answers

#### 1. Multi-Agent Competitive Environments
- **Zero-Sum Dynamics**: In our simulation, Agent A (Builder) and Agent B (Destroyer) have strictly zero-sum objectives. Every block placed increases Agent A's score by $+1$ and decreases Agent B's relative performance by $-1$.
- **Shared References**: Both agents interact via a shared environment pointer (`MultiAgentGridEnvironment`). State modifications made by one agent are immediately visible to the subsequent agent's sensor percepts.

#### 2. Episodic vs. Sequential Environments
- **Episodic**: The agent's experience is split into independent atomic episodes. Actions in one episode do not alter future episodes (e.g. image classification).
- **Sequential**: Decisions affect all future states and options. Placing a block on Step 1 permanently alters the environment unless Agent B removes it on a subsequent turn.

#### 3. Static vs. Dynamic Environments
- **Static**: The environment remains unchanged while the agent is computing its action (e.g., offline puzzle solving).
- **Dynamic**: The environment changes during deliberation or between an agent's individual turns. In our system, the environment is dynamic from Agent A's viewpoint because Agent B alters the grid between Agent A's steps.